# 🎮 Ultimate Tic-Tac-Toe Tier 6 Endgame Tablebase Solver
### Distributed Worker for Google Colab

This notebook computes canonical orbits of the **UTTT Tier 6 Endgame Tablebase (EGTB)**.

#### Key Highlights:
- **High Core Scaling**: Auto-scales worker threads across all available CPU cores.
- **Fast Ephemeral Prober**: Unpacks the Tier 7 tablebase onto Colab's high-speed local NVMe SSD (`/content/tier7`) for `< 20 ns` memory-mapped lookups (avoiding slow Drive FUSE latency).
- **Direct Google Drive Persistence**: Writes finished `.bin` slices directly to Google Drive so no computation is lost if the session disconnects.
- **Atomic Resume**: Automatically skips slices that have already been generated.

In [ ]:
# @title [1] Hardware & Environment Diagnostics
import multiprocessing
import os
import psutil

cores = multiprocessing.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"Detected CPU Cores : {cores}")
print(f"Detected System RAM: {ram_gb:.2f} GB")

!lscpu | grep -E "Model name|CPU\(s\):|Thread\(s\) per core:"
!df -h /content

In [ ]:
# @title [2] Mount Google Drive for Tablebase Persistence
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/uttt_egtb'
TIER6_DRIVE_DIR = os.path.join(DRIVE_BASE, 'tier6')
os.makedirs(TIER6_DRIVE_DIR, exist_ok=True)
print(f"Google Drive Mounted. Output slices will be saved directly to:\n  {TIER6_DRIVE_DIR}")

In [ ]:
# @title [3] Extract Tier 7 Tablebase to Fast Local Ephemeral SSD
# Lookups must run on local NVMe SSD to achieve < 20 ns memory-mapped probes
import os

ARCHIVE_PATH = os.path.join(DRIVE_BASE, 'tier7_egtb.tar.zst')
LOCAL_TIER7 = '/content/tier7'

if os.path.exists(LOCAL_TIER7) and os.path.exists(os.path.join(LOCAL_TIER7, 'manifest.json')):
    print(f"Local Tier 7 tablebase already present at {LOCAL_TIER7}.")
elif os.path.exists(ARCHIVE_PATH):
    print(f"Unpacking {ARCHIVE_PATH} to /content/ ...")
    !sudo apt-get install -y -qq zstd > /dev/null
    !tar -I zstd -xf "{ARCHIVE_PATH}" -C /content/
    print(f"Extracted Tier 7 successfully to {LOCAL_TIER7}!")
    !ls -lh {LOCAL_TIER7} | head -n 5
else:
    print(f"WARNING: Archive not found at {ARCHIVE_PATH}.")
    print("Please upload 'tier7_egtb.tar.zst' to Google Drive under 'uttt_egtb/'.")

In [ ]:
# @title [4] Clone / Pull Repository & Setup Rust Toolchain
import os

# Enter your Git repo URL or clone token below
REPO_URL = "https://github.com/USER/uttt-codingame.git" # Change to your repo or upload files

%cd /content
if not os.path.exists('/content/uttt-codingame'):
    !git clone {REPO_URL} /content/uttt-codingame || echo "If cloning fails, upload project folder to /content/uttt-codingame"

%cd /content/uttt-codingame

# Install Rust compiler
!which cargo > /dev/null || curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ["PATH"] = f"{os.path.expanduser('~')}/.cargo/bin:" + os.environ["PATH"]

# Compile solver binary in release mode
!cargo build --release --bin solve_egtb

### Orbit Catalog Reference

Choose an orbit that has not been computed yet:

| Orbit | Triplet $(j, k, l)$ | Open Center (4)? | Reachable $\tau$ Count | Recommended Worker |
|:---:|:---:|:---:|:---:|:---|
| **0** | `(0, 1, 2)` | No | 107 | ✅ **Completed** (Local Workstation) |
| **1** | `(0, 1, 3)` | No | 263 | ⏳ **In Progress** (Local Workstation) |
| **2** | `(0, 1, 5)` | No | 186 | Local Queue |
| **3** | `(0, 1, 6)` | No | 271 | Local Queue |
| **4** | `(0, 1, 7)` | No | 207 | Colab / Cloud |
| **5** | `(0, 1, 8)` | No | 255 | Colab / Cloud |
| **6** | `(0, 2, 6)` | No | 389 | 🚀 **Colab Worker 1 (Recommended)** |
| **7** | `(0, 2, 7)` | No | 203 | Colab / Cloud |
| **8** | `(0, 5, 7)` | No | 99 | Colab / Cloud |
| **9** | `(1, 3, 5)` | No | 159 | Colab / Cloud |
| **10** | `(0, 1, 4)` | Yes | 343 | 🚀 **Colab Worker 3 (Recommended)** |
| **11** | `(0, 2, 4)` | Yes | 397 | 🚀 **Colab Worker 2 (Recommended)** |
| **12** | `(0, 4, 5)` | Yes | 262 | Colab / Cloud |
| **13** | `(0, 4, 8)` | Yes | 99 | Colab / Cloud |
| **14** | `(1, 3, 4)` | Yes | 266 | Colab / Cloud |
| **15** | `(1, 4, 7)` | Yes | 78 | Colab / Cloud |

In [ ]:
# @title [5] Execute Solver for Chosen Orbit
# Change ORBIT_TO_SOLVE to the orbit you want this instance to compute
ORBIT_TO_SOLVE = 6 # @param {type:"integer"}
WORKER_THREADS = multiprocessing.cpu_count() # @param {type:"integer"}

%cd /content/uttt-codingame
!python colab/tier6_worker.py \
    --orbit {ORBIT_TO_SOLVE} \
    --threads {WORKER_THREADS} \
    --tier7 /content/tier7 \
    --out-dir {TIER6_DRIVE_DIR} \
    --max-empty 4

In [ ]:
# @title [6] Check Status & Progress
import glob

slices = glob.glob(f"{TIER6_DRIVE_DIR}/tier6_orbit_{ORBIT_TO_SOLVE}_tau_*.bin")
print(f"Completed slices for Orbit {ORBIT_TO_SOLVE} in Drive: {len(slices)}")
!ls -lh {TIER6_DRIVE_DIR}/tier6_orbit_{ORBIT_TO_SOLVE}_tau_*.bin | head -n 15